# Bootstrap, Bagging & Boosting
## Ensemble Methods & Uncertainty Quantification

### CE 315 - Junior Design

**What we've learned so far:**
- Linear models: One set of weights
- Decision trees: One tree
- Random Forest: **Many** trees voting together ← Why does this work?

**Today's big ideas:**
1. **Bootstrap**: Resample your data to measure uncertainty
2. **Bagging**: Build many models on bootstrap samples (what Random Forest does!)
3. **Boosting**: Build models sequentially, focusing on mistakes

**Why this matters:**
- Bootstrap: "How confident are we?" (critical for engineering)
- Bagging: Reduce variance, prevent overfitting
- Boosting: Often the **best** ML algorithm for tabular data

**Dataset:** Bike sharing in Washington DC
- Predict hourly bike rentals
- Real data from Capital Bikeshare system
- Features: weather, time, season, holidays

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

print("Libraries loaded")

---
# Part 1: The Dataset - Bike Sharing in DC

**Capital Bikeshare** is a bike rental system in Washington DC.

**Problem:** How many bikes should be available at each hour?
- Too few -> customers can't rent (lost revenue)
- Too many -> bikes sit unused (wasted resources)

**Our goal:** Predict hourly demand based on:
- Time (hour, day, month)
- Weather (temperature, humidity, windspeed)
- Special conditions (holiday, working day)

In [ ]:
# Load bike sharing data

# loading from UCI repository
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00275/Bike-Sharing-Dataset.zip'
import zipfile
import io
import requests

r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))
df = pd.read_csv(z.open('hour.csv'))
print(" Data loaded from UCI repository")

print(f"\n Dataset: {len(df)} hours of bike rental data")
print(f"Time period: ~{len(df)//24} days (~{len(df)//24//365} years)")

In [ ]:
# First loo
df.head(10)

In [ ]:
# Feature descriptions
print("\n Feature Descriptions:\n")
print("Time Features:")
print("  • season: 1=spring, 2=summer, 3=fall, 4=winter")
print("  • yr: 0=2011, 1=2012")
print("  • mnth: Month (1-12)")
print("  • hr: Hour (0-23)")
print("  • weekday: 0=Sunday,.., 6=Saturday")
print("  • holiday: 1=holiday, 0=not holiday")
print("  • workingday: 1=workday, 0=weekend/holiday")
print("\nWeather Features:")
print("  • weathersit: 1=Clear, 2=Mist/Cloudy, 3=Light Rain/Snow")
print("  • temp: Normalized temperature (0-1)")
print("  • atemp: Normalized 'feels like' temperature")
print("  • hum: Normalized humidity")
print("  • windspeed: Normalized wind speed")
print("\nTarget:")
print("  • cnt: Count of total rental bikes (casual + registered)")

print(f"\n Basic Statistics:")
print(f"  Average rentals per hour: {df['cnt'].mean():.0f}")
print(f"  Min rentals: {df['cnt'].min()}")
print(f"  Max rentals: {df['cnt'].max()}")
print(f"  Std dev: {df['cnt'].std():.0f}")

In [ ]:
# Do some EDA
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Hourly pattern
hourly_avg = df.groupby('hr')['cnt'].mean()
axes[0,0].plot(hourly_avg.index, hourly_avg.values, 'o-', linewidth=2, markersize=6)
axes[0,0].set_xlabel('Hour of Day')
axes[0,0].set_ylabel('Average Rentals')
axes[0,0].set_title('Rental Pattern by Hour')
axes[0,0].grid(True, alpha=0.3)
axes[0,0].axvspan(7, 9, alpha=0.2, color='red', label='Morning Rush')
axes[0,0].axvspan(17, 19, alpha=0.2, color='blue', label='Evening Rush')
axes[0,0].legend()

# Day of week pattern
day_names = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']
daily_avg = df.groupby('weekday')['cnt'].mean()
axes[0,1].bar(range(7), daily_avg.values, color=['red' if i in [0,6] else 'blue' for i in range(7)])
axes[0,1].set_xlabel('Day of Week')
axes[0,1].set_ylabel('Average Rentals')
axes[0,1].set_title('Rental Pattern by Day')
axes[0,1].set_xticks(range(7))
axes[0,1].set_xticklabels(day_names)
axes[0,1].grid(axis='y', alpha=0.3)

# Temperature effect
axes[0,2].scatter(df['temp'], df['cnt'], alpha=0.1, s=5)
axes[0,2].set_xlabel('Temperature (normalized)')
axes[0,2].set_ylabel('Rentals')
axes[0,2].set_title('Temperature vs Rentals')
axes[0,2].grid(True, alpha=0.3)

# Season pattern
season_names = ['Spring', 'Summer', 'Fall', 'Winter']
season_avg = df.groupby('season')['cnt'].mean()
axes[1,0].bar(range(1, 5), season_avg.values, color=['green', 'yellow', 'orange', 'blue'])
axes[1,0].set_xlabel('Season')
axes[1,0].set_ylabel('Average Rentals')
axes[1,0].set_title('Rental Pattern by Season')
axes[1,0].set_xticks(range(1, 5))
axes[1,0].set_xticklabels(season_names)
axes[1,0].grid(axis='y', alpha=0.3)

# Weather effect
weather_names = ['Clear', 'Cloudy', 'Rain/Snow']
weather_avg = df.groupby('weathersit')['cnt'].mean()
axes[1,1].bar(weather_avg.index, weather_avg.values, color=['green', 'gray', 'blue', 'darkblue'][:len(weather_avg)])
axes[1,1].set_xticks(weather_avg.index)
axes[1,1].set_xticklabels(['Clear', 'Cloudy', 'Rain/Snow', 'Heavy Rain'][:len(weather_avg)], rotation=15, ha='right')
axes[1,1].set_xlabel('Weather')
axes[1,1].set_ylabel('Average Rentals')
axes[1,1].set_title('Rental Pattern by Weather')
axes[1,1].set_xticks(range(1, 4))
axes[1,1].set_xticklabels(weather_names)
axes[1,1].grid(axis='y', alpha=0.3)

# Working day vs weekend
workday_avg = df.groupby('workingday')['cnt'].mean()
axes[1,2].bar([0, 1], workday_avg.values, color=['red', 'blue'])
axes[1,2].set_xlabel('Day Type')
axes[1,2].set_ylabel('Average Rentals')
axes[1,2].set_title('Working Day vs Weekend/Holiday')
axes[1,2].set_xticks([0, 1])
axes[1,2].set_xticklabels(['Weekend/Holiday', 'Working Day'])
axes[1,2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nObservations:")
print("1. Clear rush hour peaks (8am and 5-6pm)")
print("2. Higher rentals on working days (commuting)")
print("3. Temperature matters (more rides when warm)")
print("4. Bad weather reduces rentals")
print("5. Fall has highest ridership")

---
# Part 2: Bootstrap - Measuring Uncertainty

## The Problem

You fit a model and get: "Average demand = 189 bikes/hour"

**But how confident are we?**
- Is it 189 ± 5?
- Or 189 ± 50?

**In engineering, uncertainty matters**
- Nuclear reactor: ±0.1% vs ±10% is a big deal
- Bike sharing: Affects inventory decisions

## The Bootstrap Method

**Core idea:** Resample your data (with replacement) many times

**Algorithm:**
1. Original data: n samples
2. Create bootstrap sample: randomly select n samples **with replacement**
3. Calculate statistic (mean, median, model coefficients, etc.)
4. Repeat 1000+ times
5. Distribution of statistics = confidence interval!

**Why it works:**
- Simulates drawing multiple samples from population
- "The sample is to the population as the bootstrap sample is to the sample"

In [ ]:
# Simple bootstrap example: estimate mean bike rentals
sample_data = df['cnt'].values
n = len(sample_data)

# Original statistic
original_mean = sample_data.mean()
print(f"Original sample mean: {original_mean:.1f} bikes/hour")

# Bootstrap
n_bootstrap = 1000
bootstrap_means = []

for i in range(n_bootstrap):
    # Resample with replacement
    bootstrap_sample = np.random.choice(sample_data, size=n, replace=True)
    bootstrap_means.append(bootstrap_sample.mean())

bootstrap_means = np.array(bootstrap_means)

# Calculate confidence interval
ci_lower = np.percentile(bootstrap_means, 2.5)
ci_upper = np.percentile(bootstrap_means, 97.5)

print(f"\nBootstrap results ({n_bootstrap} resamples):")
print(f"  Mean of bootstrap means: {bootstrap_means.mean():.1f}")
print(f"  Standard error: {bootstrap_means.std():.1f}")
print(f"  95% Confidence Interval: [{ci_lower:.1f}, {ci_upper:.1f}]")
print(f"\n💡 We are 95% confident the true mean is between {ci_lower:.1f} and {ci_upper:.1f}")

In [ ]:
# Visualize bootstrap distribution
plt.figure(figsize=(12, 6))

plt.hist(bootstrap_means, bins=50, edgecolor='black', alpha=0.7, color='skyblue')
plt.axvline(original_mean, color='red', linestyle='--', linewidth=2, 
           label=f'Original Mean: {original_mean:.1f}')
plt.axvline(ci_lower, color='green', linestyle='--', linewidth=2, 
           label=f'95% CI: [{ci_lower:.1f}, {ci_upper:.1f}]')
plt.axvline(ci_upper, color='green', linestyle='--', linewidth=2)

# Shade the confidence interval
plt.axvspan(ci_lower, ci_upper, alpha=0.2, color='green')

plt.xlabel('Mean Rentals (bikes/hour)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Bootstrap Distribution of Mean', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print("\nInterpretation:")
print("The histogram shows all possible means from resampling.")
print("95% of bootstrap means fall in the green shaded region.")
print("This gives us confidence in our estimate!")

In [ ]:
# Bootstrap for model predictions
# Let's build a simple linear regression and bootstrap its predictions

# Prepare data
feature_cols = ['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday',
               'weathersit', 'temp', 'atemp', 'hum', 'windspeed']
X = df[feature_cols].values
y = df['cnt'].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {len(X_train)} hours")
print(f"Test set: {len(X_test)} hours")

# Function to bootstrap model predictions
def bootstrap_predictions(model_class, X_train, y_train, X_test, n_bootstrap=100):
    predictions = []
    
    for i in range(n_bootstrap):
        # Bootstrap sample
        indices = np.random.choice(len(X_train), size=len(X_train), replace=True)
        X_boot = X_train[indices]
        y_boot = y_train[indices]
        
        # Train model
        model = model_class()
        model.fit(X_boot, y_boot)
        
        # Predict
        preds = model.predict(X_test)
        predictions.append(preds)
    
    return np.array(predictions)

# Bootstrap linear regression
print("\nBootstrapping linear regression predictions...")
lr_bootstrap_preds = bootstrap_predictions(LinearRegression, X_train, y_train, X_test, n_bootstrap=100)

print(f"Created {lr_bootstrap_preds.shape[0]} different models")
print(f"Each predicts {lr_bootstrap_preds.shape[1]} test samples")

---
# Part 3: Bagging - Bootstrap Aggregating

## From Bootstrap to Bagging

**Observation:** Suppose just built 100 different models via bootstrap!

**Question:** What if we *average* their predictions?

**Answer:** That's *bagging*

## Bagging Algorithm

1. **Bootstrap**: Create B bootstrap samples
2. **Aggregate**: Train model on each bootstrap sample
3. **Predict**: Average predictions (regression) or vote (classification)

**Why it works:**
- Reduces variance (averaging smooths out fluctuations)
- Prevents overfitting
- Works best with high-variance models (like deep trees!)

**Random Forest = Bagging + Random feature selection**
- You already know about this!
- RF is bagged decision trees with extra randomness

In [ ]:
# Compare single tree vs bagging

# Single decision tree
single_tree = DecisionTreeRegressor(max_depth=10, random_state=42)
single_tree.fit(X_train, y_train)
single_tree_pred = single_tree.predict(X_test)
single_tree_mae = mean_absolute_error(y_test, single_tree_pred)
single_tree_r2 = r2_score(y_test, single_tree_pred)

print("Single Decision Tree (depth=10):")
print(f"  MAE: {single_tree_mae:.2f}")
print(f"  R²: {single_tree_r2:.3f}")

# Bagged trees (manually using our bootstrap predictions)
tree_bootstrap_preds = bootstrap_predictions(
    lambda: DecisionTreeRegressor(max_depth=10, random_state=42),
    X_train, y_train, X_test, n_bootstrap=100
)

# Average the predictions
bagged_pred = tree_bootstrap_preds.mean(axis=0)
bagged_mae = mean_absolute_error(y_test, bagged_pred)
bagged_r2 = r2_score(y_test, bagged_pred)

print("\nBagged Trees (100 trees, depth=10):")
print(f"  MAE: {bagged_mae:.2f}")
print(f"  R²: {bagged_r2:.3f}")

print(f"\n Improvement from bagging:")
print(f"  MAE reduced by: {single_tree_mae - bagged_mae:.2f} ({(1-bagged_mae/single_tree_mae)*100:.1f}%)")
print(f"  R² increased by: {bagged_r2 - single_tree_r2:.3f}")

In [ ]:
# Use sklearn's BaggingRegressor
bagging_model = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=10),
    n_estimators=100,
    random_state=42
)

bagging_model.fit(X_train, y_train)
bagging_pred_sklearn = bagging_model.predict(X_test)
bagging_mae_sklearn = mean_absolute_error(y_test, bagging_pred_sklearn)
bagging_r2_sklearn = r2_score(y_test, bagging_pred_sklearn)

print("Sklearn BaggingRegressor:")
print(f"  MAE: {bagging_mae_sklearn:.2f}")
print(f"  R²: {bagging_r2_sklearn:.3f}")
print("\n Results match our manual implementation")

In [ ]:
# Show how bagging reduces variance
# Train 20 different single trees with different random states
single_tree_maes = []
for seed in range(20):
    tree = DecisionTreeRegressor(max_depth=10, random_state=seed)
    tree.fit(X_train, y_train)
    pred = tree.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    single_tree_maes.append(mae)

# Train 20 different bagged ensembles
bagged_maes = []
for seed in range(20):
    bagged = BaggingRegressor(
        estimator=DecisionTreeRegressor(max_depth=10),
        n_estimators=100,
        random_state=seed
    )
    bagged.fit(X_train, y_train)
    pred = bagged.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    bagged_maes.append(mae)

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
axes[0].boxplot([single_tree_maes, bagged_maes], labels=['Single Tree', 'Bagged Trees'])
axes[0].set_ylabel('MAE', fontsize=12)
axes[0].set_title('Variance in Performance', fontsize=13, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Scatter plot
axes[1].scatter(range(20), single_tree_maes, s=100, alpha=0.6, label='Single Tree')
axes[1].scatter(range(20), bagged_maes, s=100, alpha=0.6, label='Bagged Trees')
axes[1].axhline(np.mean(single_tree_maes), color='blue', linestyle='--', alpha=0.5)
axes[1].axhline(np.mean(bagged_maes), color='orange', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Random Seed', fontsize=12)
axes[1].set_ylabel('MAE', fontsize=12)
axes[1].set_title('Stability Across Random Seeds', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n Statistics:")
print(f"Single Tree - Mean MAE: {np.mean(single_tree_maes):.2f}, Std: {np.std(single_tree_maes):.2f}")
print(f"Bagged Trees - Mean MAE: {np.mean(bagged_maes):.2f}, Std: {np.std(bagged_maes):.2f}")
print(f"\n Bagging reduces variance by {(1 - np.std(bagged_maes)/np.std(single_tree_maes))*100:.1f}%!")
print(" More stable, more reliable predictions.")

---
# Part 4: Random Forest Revisited

Now we understand what Random Forest **really** is.

**Random Forest = Bagging + Extra Randomness**

**Two types of randomness:**
1. **Bootstrap samples** (bagging) ← We just learned this.
2. **Random feature selection** at each split ← Extra decorrelation

**Why add feature randomness?**
- With just bagging, trees might be too similar (correlated)
- Random features → more diverse trees → better averaging

In [ ]:
# Compare Bagging vs Random Forest

# Bagging (all features at each split)
bagging = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=5),
    n_estimators=200,
    random_state=42
)
bagging.fit(X_train, y_train)
bagging_pred = bagging.predict(X_test)
bagging_mae = mean_absolute_error(y_test, bagging_pred)
bagging_r2 = r2_score(y_test, bagging_pred)

# Random Forest (random features at each split)
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=5,
    max_features=0.80,
    random_state=42
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)

print("Bagging (all features):")
print(f"  MAE: {bagging_mae:.2f}")
print(f"  R²: {bagging_r2:.3f}")

print("\nRandom Forest (random features):")
print(f"  MAE: {rf_mae:.2f}")
print(f"  R²: {rf_r2:.3f}")

print(f"\nDifference: RF is {rf_mae - bagging_mae:.2f} MAE {'better' if rf_mae < bagging_mae else 'worse'}")
print("\n Random Forest often (but not always) outperforms plain bagging.")
print("   The extra randomness helps when features are correlated.")

---
# Part 5: Boosting - A Different Ensemble Strategy

## Bagging vs Boosting

**Bagging (what we just learned):**
- Build models **in parallel** (independently)
- Each model equally weighted
- Reduces variance
- Works best with complex models (deep trees)

**Boosting (new):**
- Build models **sequentially** (each learns from previous)
- Later models focus on mistakes of earlier models
- Reduces bias AND variance
- Works best with simple models (shallow trees)

## AdaBoost Algorithm (Simplified)

1. **Start**: All samples have equal weight
2. **Train** model on weighted data
3. **Identify** mistakes: increase weight of misclassified samples
4. **Repeat**: Next model focuses more on hard cases
5. **Combine**: Weighted vote (better models get more say)

**Intuition:** Like studying for an exam
- First pass: Learn everything equally
- Second pass: Focus more on what you got wrong
- Third pass: Really drill the hard problems

## Gradient Boosting

**Even more powerful variant:**
- Each model predicts the **residuals** (errors) of previous models
- Gradually reduces error
- Think: repeatedly fixing what's wrong

**State-of-the-art implementations:**
- **XGBoost**: Extreme Gradient Boosting
- **LightGBM**: Light Gradient Boosting Machine
- **CatBoost**: Categorical Boosting

These win most Kaggle competitions!

In [ ]:
# AdaBoost Regressor
adaboost = AdaBoostRegressor(
    estimator=DecisionTreeRegressor(max_depth=10), 
    n_estimators=100,
    random_state=42
)

adaboost.fit(X_train, y_train)
ada_pred = adaboost.predict(X_test)
ada_mae = mean_absolute_error(y_test, ada_pred)
ada_r2 = r2_score(y_test, ada_pred)

print("AdaBoost (100 trees):")
print(f"  MAE: {ada_mae:.2f}")
print(f"  R²: {ada_r2:.3f}")

In [ ]:
# Gradient Boosting
gb = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=10,
    learning_rate=0.1,
    random_state=42
)

gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
gb_mae = mean_absolute_error(y_test, gb_pred)
gb_r2 = r2_score(y_test, gb_pred)

print("Gradient Boosting (100 trees):")
print(f"  MAE: {gb_mae:.2f}")
print(f"  R²: {gb_r2:.3f}")

In [ ]:
# Visualize how boosting improves over iterations
train_scores = []
test_scores = []
n_estimators_list = range(1, 101, 5)

for n_est in n_estimators_list:
    gb_temp = GradientBoostingRegressor(
        n_estimators=n_est,
        max_depth=10,
        learning_rate=0.1,
        random_state=42
    )
    gb_temp.fit(X_train, y_train)
    
    train_pred = gb_temp.predict(X_train)
    test_pred = gb_temp.predict(X_test)
    
    train_scores.append(mean_absolute_error(y_train, train_pred))
    test_scores.append(mean_absolute_error(y_test, test_pred))

plt.figure(figsize=(12, 6))
plt.plot(n_estimators_list, train_scores, 'o-', linewidth=2, markersize=6, label='Training MAE')
plt.plot(n_estimators_list, test_scores, 's-', linewidth=2, markersize=6, label='Test MAE')
plt.xlabel('Number of Trees', fontsize=12)
plt.ylabel('MAE', fontsize=12)
plt.title('Gradient Boosting: Performance vs Number of Trees', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print("\n Observations:")
print("1. Both training and test error decrease steadily")
print("2. Training error keeps going down (eventual overfitting risk)")
print("3. Test error plateaus around 50-100 trees")
print("\n Boosting gradually improves predictions by fixing errors!")

---
# Part 6: The Grand Comparison

Let's compare ALL the methods we've learned!

In [ ]:
# Train all models
models = {
    'Linear Regression': LinearRegression(),
    'Single Tree (depth=10)': DecisionTreeRegressor(max_depth=10, random_state=42),
    'Bagging (100 trees)': BaggingRegressor(
        estimator=DecisionTreeRegressor(max_depth=10),
        n_estimators=100,
        random_state=42
    ),
    'Random Forest (100 trees)': RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42
    ),
    'AdaBoost (100 trees)': AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=10),
        n_estimators=100,
        random_state=42
    ),
    'Gradient Boosting (100 trees)': GradientBoostingRegressor(
        n_estimators=100,
        max_depth=10,
        learning_rate=0.1,
        random_state=42
    )
}

results = []

for name, model in models.items():
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    
    # Evaluate
    train_mae = mean_absolute_error(y_train, train_pred)
    test_mae = mean_absolute_error(y_test, test_pred)
    test_r2 = r2_score(y_test, test_pred)
    
    results.append({
        'Model': name,
        'Train MAE': train_mae,
        'Test MAE': test_mae,
        'Test R²': test_r2
    })
    
    print(f"{name:35s} Train MAE: {train_mae:6.2f}  Test MAE: {test_mae:6.2f}  Test R²: {test_r2:.3f}")

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Test MAE')

print("\n" + "="*80)
print("\n Rankings by Test MAE (lower is better):")
print(results_df.to_string(index=False))

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# MAE comparison
x_pos = np.arange(len(results_df))
axes[0].barh(x_pos, results_df['Test MAE'], alpha=0.7, 
            color=['gold' if i == 0 else 'lightblue' for i in range(len(results_df))])
axes[0].set_yticks(x_pos)
axes[0].set_yticklabels(results_df['Model'])
axes[0].set_xlabel('Test MAE (lower is better)', fontsize=12)
axes[0].set_title('Model Comparison: MAE', fontsize=14, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)
axes[0].invert_yaxis()

# Add values
for i, v in enumerate(results_df['Test MAE']):
    axes[0].text(v + 2, i, f'{v:.2f}', va='center', fontsize=10)

# R² comparison
results_r2_sorted = results_df.sort_values('Test R²', ascending=False)
x_pos = np.arange(len(results_r2_sorted))
axes[1].barh(x_pos, results_r2_sorted['Test R²'], alpha=0.7,
            color=['gold' if i == 0 else 'lightgreen' for i in range(len(results_r2_sorted))])
axes[1].set_yticks(x_pos)
axes[1].set_yticklabels(results_r2_sorted['Model'])
axes[1].set_xlabel('Test R² (higher is better)', fontsize=12)
axes[1].set_title('Model Comparison: R²', fontsize=14, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)
axes[1].invert_yaxis()

# Add values
for i, v in enumerate(results_r2_sorted['Test R²']):
    axes[1].text(v - 0.05, i, f'{v:.3f}', va='center', ha='right', fontsize=10, color='white', fontweight='bold')

plt.tight_layout()
plt.show()

best_model = results_df.iloc[0]['Model']
print(f"\n Winner: {best_model}")
print("\n Insights:")
print("1. Ensemble methods (bagging/boosting) beat single models")
print("2. Boosting often wins on tabular data")
print("3. Linear regression struggles with non-linear patterns")
print("4. Single tree overfits despite depth limit")

---
# Part 7: When to Use What?

## Decision Framework

### Use **Bootstrap** when:
-  You need confidence intervals
-  You need to quantify uncertainty
-  Small dataset (need to squeeze out more info)
-  Any statistical estimate (mean, median, model parameters)

### Use **Bagging** when:
-  You have high-variance models (deep trees)
-  You want stable predictions
-  Features are not too correlated
-  You can train models in parallel (speed)

### Use **Random Forest** when:
-  You want bagging + extra robustness
-  Features might be correlated
-  You need feature importance
-  "Works well out of the box" solution needed
-  Interpretability less critical

### Use **Boosting** when:
-  You want maximum accuracy
-  Tabular data
-  You have time to tune hyperparameters
-  You're okay with sequential training (slower)
-  Kaggle competition!

## Practical Advice

**For most projects:**
1. Start with **Random Forest** (good baseline)
2. Try **Gradient Boosting** (often best)
3. Use **Bootstrap** for uncertainty estimates
4. Compare with **cross-validation**

**For production:**
- **Random Forest**: Fast prediction, good enough
- **Gradient Boosting**: Best accuracy, but slower
- **Bagging**: If you need custom base models

**For research/exploration:**
- **Bootstrap**: Understand uncertainty
- **Random Forest**: Feature importance
- Compare ALL methods

---
# Part 8: Feature Importance from Ensembles

In [ ]:
# Compare feature importances from different ensemble methods
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Random Forest
rf_importances = rf.feature_importances_
rf_indices = np.argsort(rf_importances)[::-1]

axes[0].barh(range(len(feature_cols)), rf_importances[rf_indices], alpha=0.7)
axes[0].set_yticks(range(len(feature_cols)))
axes[0].set_yticklabels([feature_cols[i] for i in rf_indices])
axes[0].set_xlabel('Importance', fontsize=11)
axes[0].set_title('Random Forest\nFeature Importance', fontsize=12, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)
axes[0].invert_yaxis()

# Gradient Boosting
gb_importances = gb.feature_importances_
gb_indices = np.argsort(gb_importances)[::-1]

axes[1].barh(range(len(feature_cols)), gb_importances[gb_indices], alpha=0.7, color='green')
axes[1].set_yticks(range(len(feature_cols)))
axes[1].set_yticklabels([feature_cols[i] for i in gb_indices])
axes[1].set_xlabel('Importance', fontsize=11)
axes[1].set_title('Gradient Boosting\nFeature Importance', fontsize=12, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)
axes[1].invert_yaxis()

# Side-by-side comparison of top 5
top_n = 5
rf_top = [(feature_cols[i], rf_importances[i]) for i in rf_indices[:top_n]]
gb_top = [(feature_cols[i], gb_importances[i]) for i in gb_indices[:top_n]]

x = np.arange(top_n)
width = 0.35

axes[2].bar(x - width/2, [imp for _, imp in rf_top], width, label='Random Forest', alpha=0.7)
axes[2].bar(x + width/2, [imp for _, imp in gb_top], width, label='Gradient Boosting', alpha=0.7, color='green')
axes[2].set_xlabel('Feature', fontsize=11)
axes[2].set_ylabel('Importance', fontsize=11)
axes[2].set_title('Top 5 Features\nComparison', fontsize=12, fontweight='bold')
axes[2].set_xticks(x)
axes[2].set_xticklabels([feat for feat, _ in rf_top], rotation=45, ha='right')
axes[2].legend()
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n Top 5 Most Important Features:\n")
print("Random Forest:")
for i, (feat, imp) in enumerate(rf_top, 1):
    print(f"  {i}. {feat:15s} {imp:.4f}")

print("\nGradient Boosting:")
for i, (feat, imp) in enumerate(gb_top, 1):
    print(f"  {i}. {feat:15s} {imp:.4f}")

print("\n Both models agree: hour (hr) is most important!")
print("   This makes sense - demand varies dramatically by time of day.")